In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from polypesto.core import create_sim_conditions, simulate_problem
from polypesto.core.pypesto import calculate_cis, create_ensemble, predict_with_ensemble
from polypesto.examples.base import output_dirs

# Model specific imports
from polypesto.models.binary import BinaryIrreversible
from polypesto.models.binary.utils import create_ensemble_pred_problem
from polypesto.vis import plot_ensemble_predictions

%load_ext autoreload
%autoreload 2

In [ ]:
OUTPUT_DIR = Path(
    "/Users/devoncallan/Documents/GitHub/PolyPESTO/polypesto/examples/output/simulated"
)
model = BinaryIrreversible(observables=["xA", "FA", "fA"], obs_noise=0.01)

# Define true parameters and simulation conditions
true_params = {"rA": 2.0, "rB": 1.0}
sim_conds = create_sim_conditions(
    true_params=true_params,
    conds=dict(
        A0=[0.70, 0.50],
        B0=[0.30, 0.50],
    ),
    t_evals=np.arange(0.05, 0.61, 0.05),
    meas_noise=0.01,
)

# Simulate problem and create parameter estimation problem
problem = simulate_problem(
    prob_dir=OUTPUT_DIR,
    model=model,
    conds=sim_conds,
    overwrite=True,
)

# Run parameter estimation (optimization + sampling)
result = problem.run_parameter_estimation(
    config=dict(
        optimize=dict(n_starts=50, method="Nelder-Mead"),
        sample=dict(n_samples=10000, n_chains=3),
    ),
    overwrite=True,
)

Engine will use up to 8 processes (= CPU count).



==== Running optimization ====


100%|██████████| 50/50 [00:06<00:00,  7.44it/s]
Initializing betas with "near-exponential decay".



==== Running sampling ====


100%|██████████| 10000/10000 [00:09<00:00, 1067.47it/s]
Elapsed time: 9.211473


In [ ]:
# Predict using parameter ensemble from sampling
ens_prob = create_ensemble_pred_problem(problem.paths.ensemble_dir, model)
# ens, ens_pred = problem.ensemble_prediction(ens_prob)

Engine will use up to 8 processes (= CPU count).



==== Running ensemble predictions ====


100%|██████████| 8/8 [00:24<00:00,  3.07s/it]

Error saving plot to /Users/devoncallan/Documents/GitHub/PolyPESTO/polypesto/examples/output/simulated/figures/ensemble_predictions.png: EnsemblePrediction.compute_summary.<locals>._compute_summary() missing 1 required positional argument: 'weights'


In [ ]:
ens_pred.prediction_results[0].conditions[0].output_sensi

In [3]:
import seaborn as sns
import pandas as pd


def highlight_value_above_threshold(x, threshold=1):
    return ["color: darkorange" if xi > threshold else None for xi in x]


def highlight_gradient_check(gc: pd.DataFrame):
    return (
        gc.style.apply(
            highlight_value_above_threshold,
            subset=["fd_err"],
        )
        .background_gradient(
            cmap=sns.light_palette("purple", as_cmap=True),
            subset=["abs_err"],
        )
        .background_gradient(
            cmap=sns.light_palette("red", as_cmap=True),
            subset=["rel_err"],
        )
        .background_gradient(
            cmap=sns.color_palette("viridis", as_cmap=True),
            subset=["eps"],
        )
    )


parameter_vector = problem.pypesto_problem.x_guesses[0]
gc = problem.pypesto_problem.objective.check_grad_multi_eps(
    x=parameter_vector,
    verbosity=0,
    label="rel_err",  # default
)

highlight_gradient_check(gc)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [10]:
problem.pypesto_problem.get_startpoints(n_starts=100)

array([[-0.10130563, -2.0105881 ],
       [ 1.70069211, -2.61302623],
       [-2.85642803, -2.20725953],
       [-0.43621493,  0.49089584],
       [-2.85945259,  0.64265197],
       [ 0.32607623, -2.38632237],
       [ 0.63704192, -1.48334836],
       [-0.37889808,  1.93517829],
       [ 0.35230584, -2.36322494],
       [ 0.62780243, -1.85579196],
       [-1.19269211, -0.81544917],
       [ 1.45542406, -2.89091842],
       [ 0.08655632, -0.26566506],
       [-2.79489598, -2.24636906],
       [ 1.58870891,  0.41983201],
       [ 1.40361058, -2.46903307],
       [-2.48359507, -0.07772898],
       [-2.26598139, -1.60305127],
       [ 1.6322892 , -1.62883944],
       [ 0.95836877,  1.31289687],
       [-1.97806546,  0.55380577],
       [ 0.14788463, -2.77668483],
       [-0.18359668, -2.00034351],
       [-1.77531441,  1.81606631],
       [-2.64367288, -0.27015994],
       [-1.6668413 , -2.78284478],
       [ 0.3425954 , -0.19187724],
       [ 0.09984154,  1.76619898],
       [-0.3501439 ,

In [12]:
startpoints = problem.pypesto_problem.get_startpoints(n_starts=100)
problem.pypesto_problem.objective.check_grad(
    x=startpoints[0],
    eps=1e-5,  # default
    verbosity=0,
)

,grad,fd_f,fd_b,fd_c,fd_err,abs_err,rel_err
rA,-6641.573818,-8220.991681,-5074.098934,-6647.545308,3146.892747,5.971490,0.000898
rB,36.183831,-1537.908193,1610.157004,36.124405,3148.065197,0.059425,0.001645


In [14]:
parameter_vector = problem.pypesto_problem.get_reduced_vector(startpoints[0])
problem.pypesto_problem.objective.check_grad(
    x=parameter_vector,
    eps=1e-6,
    verbosity=0,
)

,grad,fd_f,fd_b,fd_c,fd_err,abs_err,rel_err
rA,-6641.573818,-22387.695438,9092.841766,-6647.426836,31480.537205,5.853018,0.000880
rB,36.183831,-15704.203787,15776.455972,36.126092,31480.659758,0.057738,0.001598


In [15]:
gc = problem.pypesto_problem.objective.check_grad_multi_eps(
    x=parameter_vector,
    verbosity=0,
    label="rel_err",  # default
)
highlight_gradient_check(gc)

,grad,fd_f,fd_b,fd_c,fd_err,abs_err,rel_err,eps
rA,-6641.573818,-164050.739963,150755.902991,-6647.418486,314806.642955,5.844668,0.000879,0.000000
rB,36.183831,-157367.152960,157439.415557,36.131298,314806.568517,0.052532,0.001454,0.000000


In [ ]:
from pypesto import C

max_size = np.inf
x_vectors = []
x_names = [result.problem.x_names[i] for i in result.problem.x_free_indices]
abs_cutoff = np.inf
for start in result.optimize_result.list:

    if (
        start["fval"] <= abs_cutoff
        and len(x_vectors) < max_size
        # 'x' can be None if optimization failed at the startpoint
        and start["x"] is not None
    ):
        x_vectors.append(start["x"][result.problem.x_free_indices])
x_vectors = np.stack(x_vectors, axis=1)
print(x_vectors.shape)

In [ ]:
print(result.sample_result.trace_x[0, 0:, :].shape)

In [ ]:
from pypesto.ensemble.ensemble import calculate_hpd

print(result.sample_result.trace_x[0].shape)
print(calculate_hpd(result).shape)

In [ ]:
result.sample_result.trace_x[0][slice(None, None, 10)].shape

In [ ]:
# Predict using parameter ensemble from sampling
ens_prob = create_ensemble_pred_problem(problem.paths.ensemble_dir, model)
ens, ens_pred = problem.ensemble_prediction(ens_prob)

In [ ]:
ens_pred.condense_to_arrays()
ens_pred.prediction_arrays["output"].shape

In [ ]:
len(np.arange(0.01, 0.9, 0.01))

In [ ]:
ens_pred.condense_to_arrays()
ens_pred.prediction_arrays["output"].shape

In [ ]:
len(ens_pred.prediction_results)

In [ ]:
all_t = [cond.timepoints for cond in ens_pred.prediction_results[0].conditions]
sum(len(t) for t in all_t)

In [ ]:
# ens_pred.condense_to_arrays()
from pypesto.ensemble.util import get_prediction_dataset

# get_prediction_dataset(ens_pred)[:, 0, :]
get_prediction_dataset(ens_pred)

In [ ]:
ens.x_vectors

In [ ]:
ens_pred.prediction_arrays["output"]

In [ ]:
ens.check_identifiability()

In [ ]:
ens.compute_summary(percentiles_list=(5, 25, 75, 95))

In [ ]:
ens.compute_summary(percentiles_list=(5, 25, 75, 95))

In [ ]:
calculate_cis(result, ci_level=0.95)

In [ ]:
from pypesto.ensemble import (
    get_covariance_matrix_parameters,
    get_covariance_matrix_predictions,
)

get_covariance_matrix_parameters(ens)

from pypesto.visualize import parameters_correlation_matrix

In [ ]:
np.cov(ens.x_vectors)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
# plt.imshow(get_covariance_matrix_parameters(ens))
plt.imshow(np.corrcoef(ens.x_vectors))
plt.colorbar()
# ax.add_image(get_covariance_matrix_parameters(ens))

In [ ]:
np.corrcoef(ens.x_vectors)

In [ ]:
get_covariance_matrix_predictions(ens_pred)

In [ ]:
get_covariance_matrix_parameters(ens)

In [ ]:
from pypesto.visualize import ensemble_identifiability

ensemble_identifiability(ens)

In [ ]:
get_covariance_matrix_predictions(ens_pred, prediction_index=0)

In [ ]:
ens.x_vectors

In [ ]:
ens_pred.compute_summary(percentiles_list=(5, 25, 75, 95))

In [ ]:
ens_pred.prediction_arrays["output"]

In [ ]:
ens_pred.compute_chi2(problem.pypesto_problem.objective)